In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

import sys
sys.path.append('../../../')

In [10]:
import time
import warnings
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torchvision

%load_ext autoreload
%autoreload 2

from computer_vision.torch_video.parameter_parser import parser
from computer_vision.torch_video.utils.torch_utils import load_checkpoint, initialize_weights
from computer_vision.torch_video.data.dataset import NUM_CLASSES

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask'
output_dirpath=Path('D:/results/ucf101/torchvision/train')
metadata_path=data_dirpath/'metadata.pt'
class_id_path=annotation_path/"classInd.txt"

# arguments= f"""-d {root} -a {annotation_path} -m {metadata_path} -o {output_dirpath} -c {class_id_path}
# --data-fold 1 --frame-rate 8 --clip-duration 2 --step-duration 1.7 
# --num-ffmpeg-threads 16 --batch-size 16 --print-freq 100 --plot-freq 10 --time 9  --epochs 200 
# --device cuda
# """ # --use-cutmix-mixup
arguments= f"""-d {root} -a {annotation_path} -m {metadata_path} -o {output_dirpath} -c {class_id_path}
--data-fold 1 --frame-rate 4 --clip-duration 2 --step-duration 1.7 
--num-ffmpeg-threads 16 --batch-size 24 --print-freq 2 --plot-freq 2 --time 9  --n-batches 4 --epochs 4
--device cuda
""" # --use-cutmix-mixup --time 18
args=parser.parse_args(arguments.split())

In [12]:
model=torchvision.models.get_model(args.model, weights=None)
model.fc=nn.Linear(in_features=512, out_features=NUM_CLASSES, bias=True) # modify to the right number of classes
initialize_weights(model)
nn.init.normal_(model.fc.weight, mean=0.0, std=0.01) # weights are also initialized to a small Gaussian
nn.init.zeros_(model.fc.bias)

init_params={n:p.data.clone() for n, p in model.named_parameters()}

In [16]:
start_epoch,best_acc=load_checkpoint(Path(args.ouput_path)/'checkpoint'/args.last, model)

Resume training from epoch 4 with best_acc at 21.875 based on checkpoint D:\results\ucf101\torchvision\train\checkpoint\last.pth


In [17]:
for n, p in model.named_parameters():
    if torch.allclose(p.data, init_params[n]): print(n)